Pretrained DenseNet121
        ↓
Feature Extractor
        ↓
Dropout
        ↓
7-Class Classifier

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import models

In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

NUM_CLASSES = 7

print("Device:", device)
print("Number of classes:", NUM_CLASSES)

Device: cuda
Number of classes: 7


In [3]:
model = models.densenet121(
    weights=models.DenseNet121_Weights.DEFAULT
)


Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /home/sr/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100.0%


In [4]:
print(model.classifier)

Linear(in_features=1024, out_features=1000, bias=True)


In [5]:
num_features = model.classifier.in_features

model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(num_features, NUM_CLASSES)
)

In [6]:
print(model.classifier)

Sequential(
  (0): Dropout(p=0.3, inplace=False)
  (1): Linear(in_features=1024, out_features=7, bias=True)
)


In [7]:
model = model.to(device)

In [8]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Total parameters: 6,961,031
Trainable parameters: 6,961,031


In [11]:
from pathlib import Path
from torchvision import datasets, transforms

BASE_DIR = Path("..")

TRAIN_DIR = BASE_DIR / "dataset" / "train"

# Simple transform because we only need the class names here
temp_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_dataset = datasets.ImageFolder(
    root=TRAIN_DIR,
    transform=temp_transform
)

print("Classes:")
print(train_dataset.classes)

print("\nClass mapping:")
print(train_dataset.class_to_idx)

Classes:
['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']

Class mapping:
{'akiec': 0, 'bcc': 1, 'bkl': 2, 'df': 3, 'mel': 4, 'nv': 5, 'vasc': 6}


In [9]:
import pandas as pd

weights_df = pd.read_csv(
    "../splits/class_weights.csv"
)

print(weights_df)

   Class  Image_Count  Percentage  Class_Weight
0  akiec          257        3.21        4.4564
1    bcc          412        5.14        2.7798
2    bkl          891       11.11        1.2854
3     df           86        1.07       13.3173
4    mel          890       11.10        1.2868
5     nv         5367       66.95        0.2134
6   vasc          114        1.42       10.0464


In [12]:
print("Model class order:")
print(train_dataset.classes)

print("\nWeight file class order:")
print(weights_df["Class"].tolist())

Model class order:
['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']

Weight file class order:
['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']


In [13]:
class_weights = torch.tensor(
    weights_df["Class_Weight"].values,
    dtype=torch.float32
).to(device)

print("Class weights:")
print(class_weights)

Class weights:
tensor([ 4.4564,  2.7798,  1.2854, 13.3173,  1.2868,  0.2134, 10.0464],
       device='cuda:0')


In [14]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights
)


In [15]:
LEARNING_RATE = 1e-4

optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4
)

In [20]:
test_input = torch.randn(
    4,
    3,
    224,
    224
).to(device)

model.eval()

with torch.no_grad():
    output = model(test_input)

print("Input device:", test_input.device)
print("Model device:", next(model.parameters()).device)
print("Output device:", output.device)
print("Output shape:", output.shape)

Input device: cuda:0
Model device: cuda:0
Output device: cuda:0
Output shape: torch.Size([4, 7])


In [17]:
test_labels = torch.tensor(
    [0, 1, 3, 6],
    dtype=torch.long
).to(device)

loss = criterion(
    output,
    test_labels
)

print("Test loss:", loss.item())

Test loss: 2.1474671363830566


In [25]:
model.train()

test_input = torch.randn(
    4,
    3,
    224,
    224
).to(device)

test_labels = torch.tensor(
    [0, 1, 3, 6],
    dtype=torch.long
).to(device)

optimizer.zero_grad()

outputs = model(test_input)

loss = criterion(
    outputs,
    test_labels
)

loss.backward()

optimizer.step()

print("One training step completed.")
print("Loss:", loss.item())

One training step completed.
Loss: 2.1141109466552734


In [27]:
# Move model explicitly to the selected device
model.to(device)

# Verify
model_device = next(model.parameters()).device

print("Selected device:", device)
print("Actual model device:", model_device)

Selected device: cuda
Actual model device: cuda:0


In [30]:
assert NUM_CLASSES == 7
assert output.shape[1] == 7
assert class_weights.shape[0] == 7
assert next(model.parameters()).device == device

print("=" * 60)
print("PHASE 11 COMPLETED SUCCESSFULLY")
print("=" * 60)

print("Model: DenseNet121")
print("Pretrained: Yes")
print("Number of classes:", NUM_CLASSES)
print("Dropout:", 0.3)
print("Loss: Weighted CrossEntropyLoss")
print("Optimizer: AdamW")
print("Learning rate:", LEARNING_RATE)
print("Device:", device)

AssertionError: 